In [2]:
 import pandas as pd

**метрики в pandas-табличках**

Представим, что 2 добрых и очень-очень умных гнома прокрались к нам в данные, описывающие различные поездки в такси.
Далее, для каждого объекта (и его признаков, соответственно), гномы построили модели предсказания того, какая должна была быть у данных заказов длительность исполнения. Прогнозы своих моделей они положили соответственно в колонки **prediction_1** и **prediction_2**

In [3]:
df = pd.read_csv('taxi_dataset_with_predictions.csv', index_col=0)

In [4]:
df.head()

,vendor_id,pickup_datetime,passenger_count,store_and_fwd_flag,trip_duration,distance_km,prediction_1,prediction_2
id,,,,,,,,
id2875421,1,2016-03-14 17:24:55,930.399753,0,455.0,1.500479,578.156451,355.270710
id2377394,0,2016-06-12 00:43:35,930.399753,0,663.0,1.807119,962.657188,674.295781
id3858529,1,2016-01-19 11:35:24,930.399753,0,2124.0,6.392080,2546.180515,2422.132431
id3504673,1,2016-04-06 19:32:31,930.399753,0,429.0,1.487155,737.926214,795.992362
id2181028,1,2016-03-26 13:30:55,930.399753,0,435.0,1.189925,666.070794,-4.158492


<dl>
<dt> Описание колонок:
<dd>id - ID поездки </dd>
<dd>vendor_id - ID компании, осуществляющей перевозку </dd>
<dd>pickup_datetime - Таймкод начала поездки</dd>
<dd>dropoff_datetime - Таймкод конца поездки </dd>
<dd>passenger_count - Количество пассажиров </dd>
<dd>pickup_longitude - Долгота точки, в которой началась поездка </dd>
<dd>pickup_latitude - Широта точки, в которой началась поездка </dd>
<dd>dropoff_longitude - Долгота точки, в которой закончилась поездка </dd>
<dd>dropoff_latitude - Широта точки, в которой закончилась поездка </dd>
<dd>store_and_fwd_flag - Yes/No: Была ли информация сохранена в памяти транспортного средства из-за потери соединения с сервером </dd>
<dd>prediction_1 - прогнозы модели 1 <dd>
<dd>prediction_2 - прогнозы модели 2 <dd>
</dl>

Давайте попробуем оценить, насколько и вправду гномы оказались умными и осведомленными в области построения различных моделей машинного обучения, и найдем, насколько сильно их модели ошибаются на наших данных!

В начале посчитаем **MSE** модели. Для этого нужно посчитать квадратичное отклонение на каждом объекте, а потом просто усредниться! Формула представлена ниже.

$$
MSE = \frac{1}{n} \sum_i^n (a(x_i)-y_i)^2
$$

Положим результаты в переменные *error_1* и *error_2* соответственно.

In [5]:
### Реализуем переменные ошибок
error_1 = ((df['trip_duration'] - df['prediction_1'])**2).mean()
error_2 = ((df['trip_duration'] - df['prediction_2'])**2).mean()
print([int(error_1), int(error_2)])


[99994, 124936]


In [6]:
### Распечатаем значения MSE для обеих моделей.
print(f"MSE первой модели равно: {int(error_1)}")
print(f"MSE второй модели равно: {int(error_2)}")

MSE первой модели равно: 99994
MSE второй модели равно: 124936


Видно, что у MSE достаточно большой порядок. Но глазам куда будет приятнее, если мы будем считать **RMSE**:

$$
RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n} \sum_i^n (a(x_i)-y_i)^2}
$$

In [7]:
### RMSE
error_1 = ((df['trip_duration'] - df['prediction_1'])**2).mean()**0.5
error_2 = ((df['trip_duration'] - df['prediction_2'])**2).mean()**0.5
print([int(error_1), int(error_2)])

[316, 353]


In [8]:
### Распечатаем значения RMSE для обеих моделей.

print(f"RMSE первой модели равно: {int(error_1)}")
print(f"RMSE второй модели равно: {int(error_2)}")

RMSE первой модели равно: 316
RMSE второй модели равно: 353


Теперь замерим значения средней абсолютной ошибки, то есть **MAE**:

$$
MAE = \frac{1}{n} \sum_i^n |a(x_i)-y_i|
$$

Рассчитаем MAE для обеих моделей.

In [9]:
### MAE
absolute_error_1 = abs((df['trip_duration'] - df['prediction_1'])).mean()
absolute_error_2 = abs((df['trip_duration'] - df['prediction_2'])).mean()

In [10]:
### Распечатаем значения MAE для обеих моделей.
print(f"MAE первой модели равно: {int(absolute_error_1)}")
print(f"MAE второй модели равно: {int(absolute_error_2)}")

MAE первой модели равно: 300
MAE второй модели равно: 281


Мы наблюдаем ту самую ситуацию, когда, имея 2 разные модели с разными предсказаниями, финальный выбор однозначно сделать нельзя, например, сказав *"Первая модель в среднем и в общем лучше второй!"*. **Нет!** Все зависит от формы ошибки, которую мы выбираем. Иными словами, от вида той самой функции, которая наказывает наши модели и замеряет качество их прогнозов.

Ситуация, когда **MAE** и **MSE**, выбирая между 2-х,  указывают на разные модели, знакома нам еще из лекции. 

Такое может происходить, когда в одной из моделей ошибка, в среднем, независимо от порядка чисел, чуть-чуть лучше, чем во второй. Но при этом если первая модель и ошибается, то куда суровее второй. 

Давайте убедимся в том, что фатальных ошибок у второй модели больше. 

Посчитани, в скольки случаях предсказания отклоняются от ответа более, чем на **500**, для первой и второй моделей!

Назовем переменные *counter_1* и *counter_2*

In [11]:
counter_1 = ((df['trip_duration'] - df['prediction_1'])**2 >=500).sum()
counter_2 = ((df['trip_duration'] - df['prediction_2'])**2 >=500).sum()


In [12]:
### Распечатайте значения для обеих моделей.
print(f"Количество отклонений >= 500 от верного ответа для первой модели равно: {counter_1}")
print(f"Количество отклонений >= 500 от верного ответа для второй модели равно: {counter_2}")

Количество отклонений >= 500 от верного ответа для первой модели равно: 1455583
Количество отклонений >= 500 от верного ответа для второй модели равно: 1385088


**Несимметричные метрики**

Зачастую, чтобы выбрать среди всего многообразия моделей, мы можем использовать несимметричные метрики. 

**MSE** и **MAE** относятся к симметричным. Они одинаково наказывают модель как за перепредсказание, так и за недопредсказание. Ошибки *+2* и *-2* переводятся **MSE** и **MAE** в одинаковую меру: **4** в первом случае и **2** во втором.

В действительности же, можно придумать целый ряд задач, когда лучше выбирать несимметричную метрику.

Представьте, что мы - дистрибьютор инсулина, и нам нужно построить модель, которая оптимизирует поставки. В таком случае кажется, что поставить лекарства на 2 единицы больше и на 2 единицы меньше - совершенно разные сценарии и разная интерпретация катастрофичности ошибки. 

В первом случае мы можем потерять немного прибыли, а во втором - лишить пациента жизненно важного лекарства. Поэтому хотелось бы научиться еще и по-разному оценивать *недо- и перепредсказания*. Для этого и используют несимметричные метрики! Одну из них посчитаем ниже.

Рассчитаем **RMSLE**. 
$$
\text{RMSLE}(X, y, a) = \sqrt{\frac{1}{\ell}\sum_{i=1}^{\ell} \big(\log{(y_i + 1)} - \log{(a(x_i) + 1)}\big)^2}
$$

Для взятия натурального логарифма используем библиотеку **math**

P.S. Очевидно, что для некоторых отрицательных предсказаний, формула не будет работать, так как логарифм от отрицательных значений взять нельзя. Поэтому давайте подкорректируем наши прогнозы: все отрицательные числа переведем в нули (лучше уж в нашей задаче предсказать *ноль секунд*, чем *минус 100 секунд*)

Переменные с метриками назовем *rmsle_1*, *rmsle_2*.

In [14]:
### Замените все отрицательные предсказания на 0
df.loc[df['prediction_1']<0, 'prediction_1']=0
df.loc[df['prediction_2']<0, 'prediction_2']=0

In [15]:
df.head()

,vendor_id,pickup_datetime,passenger_count,store_and_fwd_flag,trip_duration,distance_km,prediction_1,prediction_2
id,,,,,,,,
id2875421,1,2016-03-14 17:24:55,930.399753,0,455.0,1.500479,578.156451,355.270710
id2377394,0,2016-06-12 00:43:35,930.399753,0,663.0,1.807119,962.657188,674.295781
id3858529,1,2016-01-19 11:35:24,930.399753,0,2124.0,6.392080,2546.180515,2422.132431
id3504673,1,2016-04-06 19:32:31,930.399753,0,429.0,1.487155,737.926214,795.992362
id2181028,1,2016-03-26 13:30:55,930.399753,0,435.0,1.189925,666.070794,0.000000


In [16]:
import math
import numpy as np

In [17]:
### Создадим переменные для лог. рмсе
rmsle_1 = (( ( (np.log(df['trip_duration']+1) - np.log(df['prediction_1']+1))**2).mean())**0.5).round(3)
rmsle_2 = (( ( (np.log(df['trip_duration']+1) - np.log(df['prediction_2']+1))**2).mean())**0.5).round(3)

In [18]:
### Распечатаем значения для обеих моделей.
print(f"RMSLE первой модели равно: {rmsle_1}")
print(f"RMSLE второй модели равно: {rmsle_2}")

RMSLE первой модели равно: 0.554
RMSLE второй модели равно: 1.556


Посчитаем, для какого количества объектов первая модель сделала перепредсказания и недопредсказания

Аналогичный расчет проведем для первой модели и недопредсказания.

оставим колонку с предсказанием такой, какой она оказалась после замены отрицательных значений

Счетчики перепредсказаний и недопредсказаний назовем *over_predicted_1* и *under_predicted_1*

In [19]:
over_predicted_1 = (df['prediction_1'] > df['trip_duration']).sum()
under_predicted_1 = (df['prediction_1'] < df['trip_duration']).sum()

In [20]:
### Распечатайте значения для обеих моделей.
print(f"Предсказания первой модели оказались больше действительных в {over_predicted_1} случаях")
print(f"Предсказания первой модели оказались меньше действительных в {under_predicted_1} случаях")

Предсказания первой модели оказались больше действительных в 1456721 случаях
Предсказания первой модели оказались меньше действительных в 1923 случаях


Аналогично для второй модели


Счетчики перепредсказаний и недопредсказаний назовем *over_predicted_2* и *under_predicted_2*

In [21]:
### Your code is here
over_predicted_2 = (df['prediction_2'] > df['trip_duration']).sum()
under_predicted_2 = (df['prediction_2'] < df['trip_duration']).sum()

In [22]:
### Распечатаем значения для обеих моделей.
print(f"Предсказания второй модели оказались больше действительных в {over_predicted_2} случаях")
print(f"Предсказания второй модели оказались меньше действительных в {under_predicted_2} случаях")

Предсказания второй модели оказались больше действительных в 811778 случаях
Предсказания второй модели оказались меньше действительных в 646866 случаях
